# 02 — Preprocessing (Stellar Classification SDSS17)

> **Fases 3-5 del playbook DS3022** — Encoding, train/test split, feature scaling.
>
> **Mejoras vs notebook base del profe (cultivos)**:
> - Drop centinelas SDSS `-9999` (detectados en notebook 01).
> - `stratify=y` en el split (Slide 11 U4_T2).
> - `StandardScaler` solo, sin doble escalado (Slide 38-40 U3_T2 — decisión cerrada con experimento previo).
> - Split ANTES de escalar para evitar data leakage (Slide 39 U3_T2 — regla de oro).

## Definition of Done (Día 2)
- [ ] CSV cargado y centinelas eliminados (99,999 filas restantes).
- [ ] Columnas metadata descartadas (queda DataFrame con 9 columnas: 8 features + target).
- [ ] Target encoded a `{GALAXY: 0, STAR: 1, QSO: 2}` (estilo profe).
- [ ] Train/test split 80/20 con `stratify=y` y `random_state=42`.
- [ ] Proporciones de clase preservadas en train y test (delta < 0.5%).
- [ ] `StandardScaler` ajustado SOLO con X_train (no data leakage).
- [ ] `X_train_scaled` tiene mean ≈ 0 (|μ| < 1e-10) y std ≈ 1.
- [ ] `X_test_scaled` tiene mean cercano a 0 PERO no exacto (sanity check del scaler).
- [ ] Artefactos serializados con joblib en `backend/models/`.
- [ ] `train_ranges.json` generado (rangos originales de features para validación de inputs en API).
- [ ] `preprocessing_summary.json` generado en `docs/`.

## Imports y configuración

In [1]:
import json
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Rutas
ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'backend' / 'data' / 'star_classification.csv'
MODELS_DIR = ROOT / 'backend' / 'models'
DATA_DIR = ROOT / 'backend' / 'data'
DOCS_DIR = ROOT / 'docs'

# Constantes
RANDOM_STATE = 42
TEST_SIZE = 0.2
SENTINEL_VALUE = -9999
PHOTOMETRIC_BANDS = ['u', 'g', 'r', 'i', 'z']

FEATURE_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
METADATA_COLS = ['obj_ID', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID',
                 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
TARGET_COL = 'class'

# Encoding estilo profe (diccionario manual — Slide del playbook)
CLASS_TO_INT = {'GALAXY': 0, 'STAR': 1, 'QSO': 2}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

print(f'ROOT: {ROOT}')
print(f'Random state: {RANDOM_STATE}, test size: {TEST_SIZE}')

ROOT: /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier
Random state: 42, test size: 0.2


## Paso 1 — Cargar CSV y limpiar centinelas SDSS

In [2]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Filas iniciales: {len(df_raw):,}')

# Drop centinelas detectados en notebook 01
mask_sentinel = (df_raw[PHOTOMETRIC_BANDS] == SENTINEL_VALUE).any(axis=1)
n_dropped = int(mask_sentinel.sum())
df = df_raw[~mask_sentinel].copy()

print(f'Filas con -9999 eliminadas: {n_dropped}')
print(f'Filas finales: {len(df):,}')

assert len(df) == len(df_raw) - n_dropped, 'Mismatch en cantidad de filas tras drop'

Filas iniciales: 100,000
Filas con -9999 eliminadas: 1
Filas finales: 99,999


## Paso 2 — Descartar columnas metadata

Las 9 columnas metadata son IDs internos del SDSS (run_ID, plate, MJD, etc.) sin valor predictivo. Mantenerlas podría introducir data leakage si correlacionan con la clase por orden de muestreo.

In [3]:
metadata_present = [c for c in METADATA_COLS if c in df.columns]
df_clean = df.drop(columns=metadata_present)

print(f'Columnas descartadas: {metadata_present}')
print(f'Columnas restantes ({len(df_clean.columns)}): {list(df_clean.columns)}')

# Validación estricta
expected_cols = set(FEATURE_COLS + [TARGET_COL])
actual_cols = set(df_clean.columns)
assert expected_cols == actual_cols, f'Columnas no coinciden. Esperado: {expected_cols}, Actual: {actual_cols}'
print('\n✓ DataFrame con exactamente 8 features + 1 target')

Columnas descartadas: ['obj_ID', 'run_ID', 'rerun_ID', 'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
Columnas restantes (9): ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'class', 'redshift']

✓ DataFrame con exactamente 8 features + 1 target


## Paso 3 — Encoding del target (estilo profe)

In [4]:
# Mapping manual (estilo notebook base del profe)
df_clean['target'] = df_clean[TARGET_COL].map(CLASS_TO_INT)

# Validar que no hay NaN tras el mapping (significaría clase inesperada)
assert df_clean['target'].notna().all(), 'Hay clases que no se mapearon — revisar CLASS_TO_INT'

df_clean['target'] = df_clean['target'].astype(int)

print('Encoding aplicado:')
for cls, code in CLASS_TO_INT.items():
    count = (df_clean['target'] == code).sum()
    print(f'  {cls} → {code}: {count:,} filas')

Encoding aplicado:
  GALAXY → 0: 59,445 filas
  STAR → 1: 21,593 filas
  QSO → 2: 18,961 filas


## Paso 4 — Train/test split (con stratify, ANTES de escalar)

**Slide 39 U3_T2 — regla de oro**: split ANTES de aplicar StandardScaler. El scaler se ajusta con la distribución del train, NUNCA debe ver el test.

In [5]:
X = df_clean[FEATURE_COLS].copy()
y = df_clean['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,  # ← mejora vs notebook cultivos del profe
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

# Validar que stratify funcionó: proporciones en train y test deben ser similares
train_prop = y_train.value_counts(normalize=True).sort_index()
test_prop = y_test.value_counts(normalize=True).sort_index()

print('\nProporción de clases:')
print('  Train: ', dict(zip([INT_TO_CLASS[i] for i in train_prop.index], train_prop.round(4).values)))
print('  Test:  ', dict(zip([INT_TO_CLASS[i] for i in test_prop.index], test_prop.round(4).values)))

max_delta = (train_prop - test_prop).abs().max()
print(f'\nMax delta proporción train vs test: {max_delta:.4f}')
assert max_delta < 0.005, f'Stratify falló — delta {max_delta:.4f} > 0.005'
print('✓ Stratify preservó las proporciones')

X_train shape: (79999, 8)
X_test shape:  (20000, 8)
y_train shape: (79999,)
y_test shape:  (20000,)

Proporción de clases:
  Train:  {'GALAXY': np.float64(0.5945), 'STAR': np.float64(0.2159), 'QSO': np.float64(0.1896)}
  Test:   {'GALAXY': np.float64(0.5944), 'STAR': np.float64(0.216), 'QSO': np.float64(0.1896)}

Max delta proporción train vs test: 0.0000
✓ Stratify preservó las proporciones


## Paso 5 — Feature Scaling (StandardScaler, fit SOLO en train)

**Decisión defendible**: StandardScaler único, no MinMax+Standard en cascada (como hacía el profe en cultivos).
- **Slide 38 U3_T2**: árboles (RF, GBoost) NO necesitan escalar. Pero LogReg/SVM/KNN/NB sí — los entrenamos a todos en notebook 03, escalado es necesario para ellos.
- **Slide 40 U3_T2**: el profe deja la elección entre MinMax/Standard abierta. Lo cerré con dato del experimento previo (cultivos): Standard solo > MinMax+Standard.

In [6]:
scaler = StandardScaler()

# fit_transform SOLO en train
X_train_scaled = scaler.fit_transform(X_train)

# Solo transform en test (NUNCA fit en test)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrame para mantener nombres de columnas (evita UserWarning del notebook base)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=FEATURE_COLS, index=X_test.index)

# Validar que el scaler funcionó como debe
train_means = X_train_scaled.mean()
train_stds = X_train_scaled.std()

print('Post-scaling X_train (debe ser μ≈0, σ≈1):')
print(f'  Means:  {train_means.round(6).to_dict()}')
print(f'  Stds:   {train_stds.round(4).to_dict()}')

# StandardScaler usa ddof=0, pandas usa ddof=1 — el delta es ~1/sqrt(N), no exactamente 1
assert train_means.abs().max() < 1e-10, f'Mean del train no es ~0: {train_means.abs().max()}'
assert (train_stds - 1).abs().max() < 0.01, f'Std del train no es ~1: {(train_stds-1).abs().max()}'
print('\n✓ Train scaling correcto (mean ≈ 0, std ≈ 1)')

# Test debe tener mean cercano pero NO exactamente 0 (el scaler no se ajustó con test)
test_means = X_test_scaled.mean()
test_stds = X_test_scaled.std()
print(f'\nPost-scaling X_test (μ cercano a 0, σ cercano a 1, NO exactos):')
print(f'  Means: {test_means.round(4).to_dict()}')
print(f'  Stds:  {test_stds.round(4).to_dict()}')

# Verificar que test NO está perfectamente escalado (sino habría leakage)
assert test_means.abs().max() < 0.5, 'Test means demasiado lejos de 0 — algo raro'
print('✓ Test scaling razonable (sin data leakage)')

Post-scaling X_train (debe ser μ≈0, σ≈1):
  Means:  {'alpha': 0.0, 'delta': -0.0, 'u': -0.0, 'g': -0.0, 'r': 0.0, 'i': -0.0, 'z': 0.0, 'redshift': -0.0}
  Stds:   {'alpha': 1.0, 'delta': 1.0, 'u': 1.0, 'g': 1.0, 'r': 1.0, 'i': 1.0, 'z': 1.0, 'redshift': 1.0}

✓ Train scaling correcto (mean ≈ 0, std ≈ 1)

Post-scaling X_test (μ cercano a 0, σ cercano a 1, NO exactos):
  Means: {'alpha': -0.0008, 'delta': -0.0007, 'u': 0.0136, 'g': 0.006, 'r': 0.0041, 'i': 0.0025, 'z': 0.0017, 'redshift': 0.0022}
  Stds:  {'alpha': 0.9994, 'delta': 1.0001, 'u': 1.0005, 'g': 0.9954, 'r': 0.9978, 'i': 0.9982, 'z': 0.9958, 'redshift': 1.004}
✓ Test scaling razonable (sin data leakage)


## Paso 6 — Generar train_ranges.json

Estos son los **rangos de features originales (NO escaladas)** del set de entrenamiento. Se usan en `/api/predict` para validar inputs del usuario y rechazar valores fuera de rango (evita extrapolación silenciosa).

In [7]:
train_ranges = {
    col: {
        'min': float(X_train[col].min()),
        'max': float(X_train[col].max()),
        'mean': float(X_train[col].mean()),
        'std': float(X_train[col].std()),
    }
    for col in FEATURE_COLS
}

ranges_path = MODELS_DIR / 'train_ranges.json'
ranges_path.write_text(json.dumps(train_ranges, indent=2))
print(f'✓ train_ranges.json escrito en {ranges_path}')

for col, r in train_ranges.items():
    print(f'  {col}: [{r["min"]:.4f}, {r["max"]:.4f}]  μ={r["mean"]:.3f}  σ={r["std"]:.3f}')

✓ train_ranges.json escrito en /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier/backend/models/train_ranges.json
  alpha: [0.0055, 359.9998]  μ=177.644  σ=96.515
  delta: [-17.6362, 83.0005]  μ=24.138  σ=19.644
  u: [10.9962, 32.7814]  μ=22.075  σ=2.251
  g: [10.4982, 31.6022]  μ=20.629  σ=2.039
  r: [9.8221, 29.5719]  μ=19.644  σ=1.856
  i: [9.4699, 32.1415]  μ=19.084  σ=1.759
  z: [9.6123, 29.3837]  μ=18.768  σ=1.767
  redshift: [-0.0100, 7.0112]  μ=0.576  σ=0.730


## Paso 7 — Serializar artefactos con joblib

In [8]:
# Scaler (lo carga el backend en /api/predict)
scaler_path = MODELS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'✓ Scaler guardado: {scaler_path} ({scaler_path.stat().st_size / 1024:.2f} KB)')

# Encoder (diccionario, para decodificar predicciones del modelo)
encoder_path = MODELS_DIR / 'label_encoder.pkl'
joblib.dump({'class_to_int': CLASS_TO_INT, 'int_to_class': INT_TO_CLASS}, encoder_path)
print(f'✓ Encoder guardado: {encoder_path} ({encoder_path.stat().st_size / 1024:.2f} KB)')

# Datasets procesados (los consume el notebook 03 — modelado)
X_train_scaled.to_parquet(DATA_DIR / 'X_train_scaled.parquet')
X_test_scaled.to_parquet(DATA_DIR / 'X_test_scaled.parquet')
X_train.to_parquet(DATA_DIR / 'X_train_raw.parquet')
X_test.to_parquet(DATA_DIR / 'X_test_raw.parquet')
y_train.to_frame('target').to_parquet(DATA_DIR / 'y_train.parquet')
y_test.to_frame('target').to_parquet(DATA_DIR / 'y_test.parquet')

print(f'\n✓ Datasets guardados en {DATA_DIR}')
for fname in ['X_train_scaled', 'X_test_scaled', 'X_train_raw', 'X_test_raw', 'y_train', 'y_test']:
    p = DATA_DIR / f'{fname}.parquet'
    print(f'  - {fname}.parquet ({p.stat().st_size / 1024:.1f} KB)')

✓ Scaler guardado: /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier/backend/models/scaler.pkl (1.15 KB)
✓ Encoder guardado: /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier/backend/models/label_encoder.pkl (0.09 KB)



✓ Datasets guardados en /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier/backend/data
  - X_train_scaled.parquet (6573.7 KB)
  - X_test_scaled.parquet (1660.7 KB)
  - X_train_raw.parquet (6540.0 KB)
  - X_test_raw.parquet (1648.0 KB)
  - y_train.parquet (531.9 KB)
  - y_test.parquet (132.1 KB)


## Paso 8 — Generar preprocessing_summary.json

In [9]:
# Hash del CSV de entrada (para reproducibilidad)
csv_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

summary = {
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'dataset_input': {
        'path': str(DATA_PATH.relative_to(ROOT)),
        'sha256': csv_hash,
        'rows_initial': len(df_raw),
    },
    'cleaning': {
        'sentinel_rows_dropped': n_dropped,
        'metadata_columns_dropped': metadata_present,
        'rows_after_cleaning': len(df_clean),
    },
    'encoding': {
        'strategy': 'manual_dict',
        'class_to_int': CLASS_TO_INT,
    },
    'split': {
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
        'stratify': True,
        'train_rows': len(X_train),
        'test_rows': len(X_test),
        'train_class_proportions': {
            INT_TO_CLASS[i]: float(p) for i, p in train_prop.items()
        },
        'test_class_proportions': {
            INT_TO_CLASS[i]: float(p) for i, p in test_prop.items()
        },
    },
    'scaling': {
        'scaler': 'StandardScaler',
        'fit_on': 'X_train only',
        'train_mean_max_abs': float(train_means.abs().max()),
        'train_std_min': float(train_stds.min()),
        'train_std_max': float(train_stds.max()),
        'test_mean_max_abs': float(test_means.abs().max()),
    },
    'artifacts': {
        'scaler': 'backend/models/scaler.pkl',
        'encoder': 'backend/models/label_encoder.pkl',
        'train_ranges': 'backend/models/train_ranges.json',
        'X_train_scaled': 'backend/data/X_train_scaled.parquet',
        'X_test_scaled': 'backend/data/X_test_scaled.parquet',
        'X_train_raw': 'backend/data/X_train_raw.parquet',
        'X_test_raw': 'backend/data/X_test_raw.parquet',
        'y_train': 'backend/data/y_train.parquet',
        'y_test': 'backend/data/y_test.parquet',
    },
    'decisions': [
        'Drop 1 fila con valor centinela -9999 (detectado en notebook 01, clase STAR).',
        'Descartar 9 columnas metadata (IDs internos del SDSS — sin valor predictivo, riesgo de leakage).',
        'Encoding manual estilo profe: {GALAXY: 0, STAR: 1, QSO: 2}.',
        f'Train/test split 80/20 con stratify=y, random_state={RANDOM_STATE}.',
        'StandardScaler único (sin doble escalado del notebook base). Ajustado SOLO en train.',
        'train_ranges.json guarda rangos de features ORIGINALES (no escaladas) para validar inputs en /api/predict.',
    ],
}

OUTPUT_PATH = DOCS_DIR / 'preprocessing_summary.json'
OUTPUT_PATH.write_text(json.dumps(summary, indent=2))
print(f'✓ preprocessing_summary.json guardado en {OUTPUT_PATH}')
print(json.dumps(summary, indent=2))

✓ preprocessing_summary.json guardado en /Users/alejandromarcelo/Desktop/PROYECTOS_2026/stellar-classifier/docs/preprocessing_summary.json
{
  "timestamp": "2026-05-27T15:21:58.283137Z",
  "dataset_input": {
    "path": "backend/data/star_classification.csv",
    "sha256": "faefcad9b950309282024dd351289ee14ae30f65a7b2ef32c9fa442da9dbe585",
    "rows_initial": 100000
  },
  "cleaning": {
    "sentinel_rows_dropped": 1,
    "metadata_columns_dropped": [
      "obj_ID",
      "run_ID",
      "rerun_ID",
      "cam_col",
      "field_ID",
      "spec_obj_ID",
      "plate",
      "MJD",
      "fiber_ID"
    ],
    "rows_after_cleaning": 99999
  },
  "encoding": {
    "strategy": "manual_dict",
    "class_to_int": {
      "GALAXY": 0,
      "STAR": 1,
      "QSO": 2
    }
  },
  "split": {
    "test_size": 0.2,
    "random_state": 42,
    "stratify": true,
    "train_rows": 79999,
    "test_rows": 20000,
    "train_class_proportions": {
      "GALAXY": 0.5944574307178839,
      "STAR": 0.21

## Resumen

| Métrica | Valor |
|---|---|
| Filas iniciales | 100,000 |
| Filas tras drop de centinelas | 99,999 |
| Columnas finales | 9 (8 features + 1 target) |
| Train rows | 79,999 |
| Test rows | 20,000 |
| Stratify | ✓ |
| Scaler | StandardScaler |
| Train mean post-scaling | ≈ 0 |
| Train std post-scaling | ≈ 1 |

## Siguiente paso
Continuar con `03_modeling.ipynb`: baseline trivial + entrenar 10 modelos con CV + classification_report + confusion matrix → elegir modelo final.